# Lab 2 — Partner B: explain **one prediction** (local SHAP)

You own the **local** question: pick a single hour and explain **why the model predicted what it did**
for *that* hour - the kind of answer you'd give a stakeholder who asks "why is your number so high?"

Your night:
1. Run the setup + SHAP setup (given)
2. Check how good the model is with a **predicted-vs-actual** plot (given)
3. **Write one line** to make a **SHAP waterfall** for the peak-demand hour (your only coding task)
4. Ship the two PNGs + this notebook through a pull request on branch `dev-local`

> Type the one SHAP line yourself.

In [ ]:
# --- setup: install SHAP, load the data, fit the model (just run this) ---
!pip install shap -q

import pandas as pd, numpy as np, matplotlib.pyplot as plt, shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

url = "https://raw.githubusercontent.com/drdave-teaching/opim5512-lab2-template/main/data/energy_model_data.csv"
df = pd.read_csv(url, parse_dates=["hour"])

FEATURES = ["temp_f", "hour_of_day", "dewpoint_f", "humidity_pct", "wind_kt", "weekend"]
X, y = df[FEATURES], df["load_mw"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print(f"model R2 (test): {r2_score(yte, model.predict(Xte)):.2f}   |   typical miss: {mean_absolute_error(yte, model.predict(Xte)):,.0f} MW")
X.head()

### How the model was built (read this — don't code it tonight)

We reused the joined weather + demand data **you built in Lab 1** — one row per hour — and trained a
small **random forest** to predict New England electricity demand from six features. It's accurate
(R^2 around 0.9), which makes it worth asking the real question of Module 2: **how does it decide?**

| feature | meaning | units |
|---|---|---|
| `temp_f` | air temperature | deg F |
| `hour_of_day` | 0-23, the hour the reading begins | hour |
| `dewpoint_f` | dew point (muggy-ness) | deg F |
| `humidity_pct` | relative humidity | % |
| `wind_kt` | wind speed | knots |
| `weekend` | 1 on Sat/Sun, else 0 | 0/1 |

**SHAP** answers "how did the model decide" by giving every feature, for every prediction, a number
in **MW**: how much it pushed that prediction **up (+)** or **down (-)** from the average. Add them
all up and you get the model's prediction. That's the whole idea.

In [ ]:
# --- SHAP setup (just run this): explain every prediction the model makes ---
explainer = shap.TreeExplainer(model)
shap_values = explainer(X)          # one row of SHAP values per hour, one column per feature
print("SHAP ready:", shap_values.shape, "(hours x features)")

## 1. Is the model any good? (given) predicted vs actual

Run this - it saves `predicted_vs_actual.png`. Points on the diagonal = perfect predictions.

In [ ]:
pred = model.predict(X)
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y, pred, alpha=0.4)
lims=[y.min(), y.max()]; ax.plot(lims, lims, "k--", lw=1)
ax.set_xlabel("actual demand (MW)"); ax.set_ylabel("predicted demand (MW)")
ax.set_title("Predicted vs actual demand")
fig.savefig("predicted_vs_actual.png", dpi=150, bbox_inches="tight")
print("saved predicted_vs_actual.png")

## 2. Your turn: a SHAP waterfall for the peak hour (one line)

The busiest hour of the month is the one to explain. This finds it for you:

```
i = int(np.argmax(model.predict(X)))          # the hour the model predicts highest
print("explaining:", df.loc[i, "hour"], "| actual:", round(df.loc[i,"load_mw"]), "MW")
```

Now write the waterfall for that hour and save it:

```
shap.plots.waterfall(shap_values[i], show=False)
plt.gcf().savefig("shap_local.png", dpi=150, bbox_inches="tight"); plt.close()
```

Then **look at it:** which features pushed this prediction **up** (red) and which pulled it **down**
(blue)? Is it mostly the hour of day, the weather, or both? That sentence is your finding.

In [ ]:
# TODO: find the peak hour (i), print it, then make the waterfall -> shap_local.png



### Download both PNGs to your laptop

In [ ]:
import os
from google.colab import files
for p in ['predicted_vs_actual.png', 'shap_local.png']:
    if os.path.exists(p):
        files.download(p)
    else:
        print(f"{p} not found yet - run the cell that makes it, then re-run this one.")

## Ship it (this is the git half of the lab)

1. **This notebook -> GitHub:** **File -> Save** -> your repo, branch **`dev-local`**, keep the path
   `notebooks/Lab2_B_Local_SHAP.ipynb`, real commit message. (Plain **Save** commits to GitHub because you opened it
   *from* GitHub. Ctrl+S only autosaves to Drive.)
2. **The two PNGs -> your repo:** the download cell put them in your Downloads folder. GitHub Desktop
   -> **Repository -> Show in Explorer** -> drag `predicted_vs_actual.png` and `shap_local.png` into **`images/`**
   (rename any `(1)` copy back first).
3. GitHub Desktop: on branch **`dev-local`** -> **Commit** (real message) -> **Push** ->
   github.com **Compare & pull request** -> ask your partner to review.